# 04 — Cross-domain Evaluation (M6)

Run the **same** EfficientNet-B0 (trained on controlled PlantVillage leaves) on **CD&S field photos** — handheld iPhone shots from Purdue ACRE, natural backgrounds. Only the two diseases that overlap both datasets are evaluated: **northern_leaf_blight** and **gray_leaf_spot**. CD&S has no healthy and no common_rust class, so this is a 2-class field test by construction.

**This is the point of the whole project.** Preprocessing is byte-for-byte the same as in-domain (the weights' own `.transforms()`, invariant #4), so the only thing that changed is the *domain*. Whatever accuracy we lose here is the honest domain gap.

> **Runs locally on CPU** — inference only. Needs:
> - `outputs/checkpoints/best.pt` (your Colab-trained model), and
> - `data/external/cds_overlap/{northern_leaf_blight,gray_leaf_spot}/` (built by `python scripts/download_cds.py --verify`).
>
> ✅ **Invariant #5:** this notebook reports controlled **and** field results together. The gap — not the in-domain number — is the finding.

## 1. Setup and guards

In [ ]:
import os
import sys
from pathlib import Path

# Local Jupyter starts in notebooks/ — step up to the repo root.
if Path.cwd().name == "notebooks":
    os.chdir("..")
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ckpt = PROJECT_ROOT / "outputs/checkpoints/best.pt"
overlap = PROJECT_ROOT / "data/external/cds_overlap"
in_domain = PROJECT_ROOT / "outputs/metrics_in_domain.json"
print("Project root:      ", PROJECT_ROOT)
print("checkpoint present:", ckpt.exists())
print("cds_overlap present:", overlap.exists())
print("M5 metrics present:", in_domain.exists(), "(needed for the gap table)")
if not ckpt.exists():
    print("\n-> Train on Colab (notebook 02); place best.pt at outputs/checkpoints/best.pt.")
if not overlap.exists():
    print("\n-> Build overlap dir: python scripts/download_cds.py --verify")

## 2. Run the cross-domain evaluation

`cross_evaluate.main()` loads `best.pt`, runs every CD&S overlap image through the model in **one forward pass**, and reports it **two ways**:

1. **Open 4-way** — the model may emit any of its 4 trained classes. Any prediction landing in healthy/common_rust is counted as a **leakage error mode** (a diseased field leaf called healthy is the worst case).
2. **Restricted 2-class** — the decision is forced to the NLB-vs-GLS logits only: "in the field, can it at least tell the two apart?"

It also prints the **domain gap** (controlled M5 vs field M6, per-class recall) and writes `outputs/metrics_cross_domain.json` + two confusion figures.

In [ ]:
from src.maize_detection.cross_evaluate import main as run_cross_eval
run_cross_eval()

## 3. Confusion matrices — restricted (2-class) and open (4-way, shows leakage)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

figs = [
    PROJECT_ROOT / "outputs/figures/confusion_matrix_cross_restricted.png",
    PROJECT_ROOT / "outputs/figures/confusion_matrix_cross_open4way.png",
]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, p in zip(axes, figs):
    ax.imshow(Image.open(p))
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. What these numbers mean

On our run the model drops from **98.4% in-domain accuracy** to **~62% field accuracy** (restricted 2-class) — and under the open 4-way view it calls roughly **a quarter of diseased field leaves "healthy"** (the leakage tally). That high false-negative behavior, not the headline accuracy, is the result.

- **Per-class recall collapses** on field images: both NLB and GLS lose ~0.3–0.4 recall versus controlled. The model learned controlled-image shortcuts (uniform backgrounds, lighting, framing) that don't transfer.
- **Two views, two questions.** The *open* view exposes how the model behaves in deployment (free to predict anything, and it leaks badly into *healthy*). The *restricted* view isolates the genuine NLB-vs-GLS signal that survives the domain shift.
- **Benchmark check.** The published CD&S study reports ~94% → ~55% for gray leaf spot. Our gap is of the same character — a transparent reproduction, not a tuned target.
- **common_rust caveat** still applies to the *controlled* side of the comparison (`data.COMMON_RUST_LEAKAGE_CAVEAT`): its near-perfect in-domain recall may be optimistic. common_rust isn't in CD&S, so it can't be field-tested here — but the model still misfires *into* common_rust on a few field leaves.

**Takeaway:** a classifier that looks near-perfect on controlled images is **not field-ready**. Measuring and reporting that gap honestly — leading with the gap, with per-class false-negative rates — is exactly what V1 set out to do.